In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import string
import os

# Force file paths to be relative to the script location
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

INPUT_TEXT_FILE = os.path.join(BASE_DIR, 'bee movie script.txt')

# Default Enigma configuration
DEFAULT_ROTOR_ORDER = ['III', 'II', 'I']
DEFAULT_POSITIONS   = [0, 0, 0]

# Training configuration
MESSAGE_LENGTH       = 100
NUM_TRAINING_SAMPLES = 5000
NUM_EPOCHS           = 20
BATCH_SIZE           = 32
LEARNING_RATE        = 0.001

# Fixed model architecture
NUM_CONV_LAYERS = 3
CONV_CHANNELS   = 512
FC_NODES        = 1024


# ============================================================================
# ENIGMA SIMULATOR
# ============================================================================
class SimpleEnigma:
    ROTORS = {
        'I':   'EKMFLGDQVZNTOWYHXUSPAIBRCJ',
        'II':  'AJDKSIRUXBLHWTMCQGZNPYFVOE',
        'III': 'BDFHJLCPRTXVZNYEIWGAKMUSQO',
        'IV':  'ESOVPZJAYQUIRHXLNFTGKDCMWB',
        'V':   'VZBRGITYUPSDNHLXAWMJQOFECK',
    }
    NOTCHES   = {'I': 'Q', 'II': 'E', 'III': 'V', 'IV': 'J', 'V': 'Z'}
    REFLECTOR = 'YRUHQSLDPXNGOKMIEBFZCWVJAT'

    def __init__(self, rotors, positions):
        self.rotors            = [self.ROTORS[r] for r in rotors]
        self.rotor_names       = rotors
        self.positions         = positions.copy()
        self.initial_positions = positions.copy()

    def reset(self):
        self.positions = self.initial_positions.copy()

    def set_positions(self, positions):
        self.positions = positions.copy()

    def get_positions(self):
        return self.positions.copy()

    def step_rotors(self):
        if self.rotor_at_notch(1):
            self.positions[1] = (self.positions[1] + 1) % 26
            self.positions[2] = (self.positions[2] + 1) % 26
        elif self.rotor_at_notch(0):
            self.positions[1] = (self.positions[1] + 1) % 26
        self.positions[0] = (self.positions[0] + 1) % 26

    def rotor_at_notch(self, rotor_index):
        notch     = self.NOTCHES[self.rotor_names[rotor_index]]
        notch_pos = ord(notch) - ord('A')
        return self.positions[rotor_index] == notch_pos

    def encrypt_char(self, char):
        if char not in string.ascii_uppercase:
            return char
        self.step_rotors()
        pos = ord(char) - ord('A')
        for i in range(3):
            pos = (pos + self.positions[i]) % 26
            pos = ord(self.rotors[i][pos]) - ord('A')
            pos = (pos - self.positions[i]) % 26
        pos = ord(self.REFLECTOR[pos]) - ord('A')
        for i in range(2, -1, -1):
            pos = (pos + self.positions[i]) % 26
            pos = self.rotors[i].index(chr(pos + ord('A')))
            pos = (pos - self.positions[i]) % 26
        return chr(pos + ord('A'))

    def encrypt(self, text):
        return ''.join(self.encrypt_char(c) for c in text.upper())


# ============================================================================
# DATA PREPARATION
# ============================================================================
def read_text_from_file(filename):
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Input file '{filename}' not found!")
    letters = []
    with open(filename, 'r') as f:
        for line in f:
            for char in line.strip().upper():
                if char in string.ascii_uppercase:
                    letters.append(char)
    text = ''.join(letters)
    print(f"Read {len(text)} letters from '{filename}'")
    return text


def generate_dataset_continuous(filename, rotor_order, initial_positions,
                                 num_samples=5000, message_length=100, step_size=1):
    full_plaintext = read_text_from_file(filename)
    total_needed   = (num_samples * step_size) + message_length
    if len(full_plaintext) < total_needed:
        repeats        = (total_needed // len(full_plaintext)) + 1
        full_plaintext = full_plaintext * repeats

    enigma          = SimpleEnigma(rotor_order, initial_positions)
    full_ciphertext = enigma.encrypt(full_plaintext)
    enigma.set_positions(initial_positions)

    data = []
    for i in range(num_samples):
        start_idx       = i * step_size
        plaintext       = full_plaintext[start_idx : start_idx + message_length]
        ciphertext      = full_ciphertext[start_idx : start_idx + message_length]
        chunk_positions = enigma.get_positions()
        for char in full_plaintext[start_idx : start_idx + step_size]:
            if char in string.ascii_uppercase:
                enigma.step_rotors()
        data.append({
            'ciphertext': ciphertext,
            'plaintext':  plaintext,
            'rotors':     rotor_order,
            'positions':  chunk_positions,
        })
    return data


# ============================================================================
# PYTORCH DATASET
# ============================================================================
class EnigmaDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item           = self.data[idx]
        ciphertext_enc = self.encode_text(item['ciphertext'])
        positions      = torch.tensor(item['positions'], dtype=torch.long)
        return ciphertext_enc, positions

    @staticmethod
    def encode_text(text):
        encoded = torch.zeros(len(text), 26)
        for i, char in enumerate(text):
            if char in string.ascii_uppercase:
                encoded[i, ord(char) - ord('A')] = 1
        return encoded


# ============================================================================
# NEURAL NETWORK
# Architecture:
#   - 3 x Conv1d blocks (512 channels each, flat), each followed by MaxPool1d
#   - 1 x FC hidden layer (1024 nodes)
#   - 3 x output heads (one per rotor, 26 classes each)
# ============================================================================
class EnigmaRotorClassifier(nn.Module):
    def __init__(self, message_length=100):
        super().__init__()

        self.pool    = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.3)

        # --- 3 conv blocks, all with 512 channels ---
        self.conv_blocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(26  if i == 0 else 512, 512, kernel_size=3, padding=1),
                nn.BatchNorm1d(512),
                nn.ReLU(),
            )
            for i in range(NUM_CONV_LAYERS)
        ])

        # Compute flattened size after conv + pool passes
        seq_len = message_length
        for _ in range(NUM_CONV_LAYERS):
            if seq_len >= 2:
                seq_len = seq_len // 2
        conv_output_size = seq_len * CONV_CHANNELS   # 12 * 512 = 6144

        # --- 1 FC hidden layer (1024 nodes) ---
        self.fc_hidden = nn.Sequential(
            nn.Linear(conv_output_size, FC_NODES),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # --- Output heads (one per rotor, 26 possible positions) ---
        self.rotor1_head = nn.Linear(FC_NODES, 26)
        self.rotor2_head = nn.Linear(FC_NODES, 26)
        self.rotor3_head = nn.Linear(FC_NODES, 26)

    def forward(self, x):
        x = x.transpose(1, 2)          # (B, 26, L)
        for block in self.conv_blocks:
            x = block(x)
            if x.size(2) >= 2:
                x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc_hidden(x)
        return self.rotor1_head(x), self.rotor2_head(x), self.rotor3_head(x)


# ============================================================================
# TRAINING
# ============================================================================
def train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS, device='cpu'):
    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3)
    best_val_acc = 0.0

    for epoch in range(num_epochs):
        # ---- Training ----
        model.train()
        train_correct = [0, 0, 0]
        train_total   = 0
        train_loss    = 0.0

        for ciphertext, positions in train_loader:
            ciphertext = ciphertext.to(device)
            positions  = positions.to(device)
            optimizer.zero_grad()
            o1, o2, o3 = model(ciphertext)
            loss = (criterion(o1, positions[:, 0])
                  + criterion(o2, positions[:, 1])
                  + criterion(o3, positions[:, 2]))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            for k, out in enumerate([o1, o2, o3]):
                train_correct[k] += (out.argmax(1) == positions[:, k]).sum().item()
            train_total += positions.size(0)

        # ---- Validation ----
        model.eval()
        val_correct = [0, 0, 0]
        val_total   = 0
        val_loss    = 0.0

        with torch.no_grad():
            for ciphertext, positions in val_loader:
                ciphertext = ciphertext.to(device)
                positions  = positions.to(device)
                o1, o2, o3 = model(ciphertext)
                loss = (criterion(o1, positions[:, 0])
                      + criterion(o2, positions[:, 1])
                      + criterion(o3, positions[:, 2]))
                val_loss += loss.item()
                for k, out in enumerate([o1, o2, o3]):
                    val_correct[k] += (out.argmax(1) == positions[:, k]).sum().item()
                val_total += positions.size(0)

        train_acc = [(c / train_total) * 100 for c in train_correct]
        val_acc   = [(c / val_total)   * 100 for c in val_correct]
        avg_val   = sum(val_acc) / 3
        scheduler.step(val_loss)

        print(f'  Epoch {epoch+1}/{num_epochs} | '
              f'Train Loss: {train_loss/len(train_loader):.4f} | '
              f'Train Acc R1:{train_acc[0]:.1f}% R2:{train_acc[1]:.1f}% R3:{train_acc[2]:.1f}% | '
              f'Val Acc R1:{val_acc[0]:.1f}% R2:{val_acc[1]:.1f}% R3:{val_acc[2]:.1f}% '
              f'(avg {avg_val:.1f}%)')

        if avg_val > best_val_acc:
            best_val_acc = avg_val
            torch.save(model.state_dict(), 'best_enigma_model.pth')
            print(f'  ✓ New best model saved (avg val acc: {best_val_acc:.2f}%)')

    return best_val_acc


# ============================================================================
# MAIN
# ============================================================================
if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')

    print('\n' + '=' * 60)
    print('MODEL ARCHITECTURE')
    print('=' * 60)
    print(f'  Conv layers  : {NUM_CONV_LAYERS} (flat, {CONV_CHANNELS} channels each)')
    print(f'  Channel flow : 26 (input) → 512 → 512 → 512')
    print(f'  FC layer     : {FC_NODES} nodes')
    print(f'  Output heads : 3 x 26 (one per rotor)')

    print('\n' + '=' * 60)
    print('GENERATING DATASET')
    print('=' * 60)

    try:
        dataset = generate_dataset_continuous(
            filename=INPUT_TEXT_FILE,
            rotor_order=DEFAULT_ROTOR_ORDER,
            initial_positions=DEFAULT_POSITIONS,
            num_samples=NUM_TRAINING_SAMPLES,
            message_length=MESSAGE_LENGTH,
            step_size=1,
        )
    except FileNotFoundError as e:
        raise FileNotFoundError(
            f"Required input file '{INPUT_TEXT_FILE}' not found. Please create it first."
        ) from e

    split_idx     = int(0.8 * len(dataset))
    train_dataset = EnigmaDataset(dataset[:split_idx])
    val_dataset   = EnigmaDataset(dataset[split_idx:])
    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print(f'Training samples   : {len(train_dataset)}')
    print(f'Validation samples : {len(val_dataset)}')

    model        = EnigmaRotorClassifier(message_length=MESSAGE_LENGTH)
    total_params = sum(p.numel() for p in model.parameters())
    print(f'Total parameters   : {total_params:,}')

    print('\n' + '=' * 60)
    print('TRAINING')
    print('=' * 60)

    best_acc = train_model(model, train_loader, val_loader, device=device)

    print('\n' + '=' * 60)
    print('TRAINING COMPLETE')
    print('=' * 60)
    print(f'Best avg validation accuracy: {best_acc:.2f}%')
    print('Model saved to: best_enigma_model.pth')

Using device: cuda

MODEL ARCHITECTURE
  Conv layers  : 3 (flat, 512 channels each)
  Channel flow : 26 (input) → 512 → 512 → 512
  FC layer     : 1024 nodes
  Output heads : 3 x 26 (one per rotor)

GENERATING DATASET
Read 37001 letters from 'C:\Users\acool\1 Capstone\Engima-Capstone-main\bee movie script.txt'
Training samples   : 4000
Validation samples : 1000
Total parameters   : 7,989,838

TRAINING
  Epoch 1/20 | Train Loss: 6.4004 | Train Acc R1:5.9% R2:46.0% R3:73.1% | Val Acc R1:4.2% R2:3.2% R3:0.1% (avg 2.5%)
  ✓ New best model saved (avg val acc: 2.50%)
  Epoch 2/20 | Train Loss: 2.3812 | Train Acc R1:32.9% R2:88.5% R3:95.0% | Val Acc R1:2.4% R2:8.9% R3:0.1% (avg 3.8%)
  ✓ New best model saved (avg val acc: 3.80%)
  Epoch 3/20 | Train Loss: 1.5025 | Train Acc R1:56.7% R2:90.3% R3:95.4% | Val Acc R1:5.4% R2:9.3% R3:0.1% (avg 4.9%)
  ✓ New best model saved (avg val acc: 4.93%)
  Epoch 4/20 | Train Loss: 1.1444 | Train Acc R1:68.7% R2:91.6% R3:96.2% | Val Acc R1:6.6% R2:9.8% R3:0.